In [ ]:
!pip install konlpy

In [ ]:
!apt update

In [ ]:
!apt install -y openjdk-17-jdk

In [ ]:
!pip install pandas numpy

In [ ]:
from konlpy.tag import Okt
okt = Okt()

In [ ]:
import pandas as pd

In [ ]:
import re
df = pd.read_csv('./daum_movie_review.csv')
df['target'] = df['rating'].apply(lambda x : 1 if x>0.5 else 0)
df['clean'] = df['review'].apply(lambda x : re.sub(r'[^가-힣\s]','',x) )                               



In [ ]:
df['clean'][0]

In [ ]:
okt = Okt()
def kor_tokenizer(text):    
    return [
        word for word, pos in okt.pos(text,stem=True) 
            if pos in ['Noun','Verb','Adjective'] and len(word) >=2        
           ]

In [ ]:
# 시간순서가 중요하므로 사용할수 없음
# from sklearn.feature_extraction.text import TfidfVectorizer
# tfidf = TfidfVectorizer(tokenizer=kor_tokenizer,max_features=10000)
# x = tfidf.fit_transform(df['clean'])

In [ ]:
kor_tokenizer(df['clean'][0]),kor_tokenizer(df['clean'][1])

In [28]:
# 단어사전 구축
from collections import Counter
vocab = Counter()
for text in df['clean']:
    vocab.update(kor_tokenizer(text))

In [40]:
# 10000개의 단어로만 구성
vocab_size = 10000
word_to_index = {
    word:idx+2 for idx, (word,seq) in enumerate(vocab.most_common(vocab_size))
}
word_to_index['<PAD>'] = 0
word_to_index['<UNK>'] = 1

def word2Sequence(text):
    return [word_to_index.get(word,1) for word in kor_tokenizer(text)]    


In [50]:
kor_tokenizer(df['clean'][0])

['들이다', '보다', '내내', '하품']

In [51]:
word2Sequence(df['clean'][0])

[345, 4, 147, 731]

In [52]:
index_to_word = {idx : word for word, idx in word_to_index.items()}
index_to_word.get(345)

'들이다'

In [63]:
# padding
import torch
from torch.nn.utils.rnn import pad_sequence
x1 = word2Sequence(df['clean'][0])
x2 = word2Sequence(df['clean'][1])
x1_tensor = torch.LongTensor(x1)
x2_tensor = torch.LongTensor(x2)
print(x1_tensor,x2_tensor)
pad_sequence([x1_tensor,x2_tensor], batch_first=True, padding_value=0)

tensor([345,   4, 147, 731]) tensor([ 123,    2,    7,  453,   16,    2,  464, 1692, 2089, 3174])


tensor([[ 345,    4,  147,  731,    0,    0,    0,    0,    0,    0],
        [ 123,    2,    7,  453,   16,    2,  464, 1692, 2089, 3174]])

In [ ]:
MAX_LEN = 500
def padding(texts):
    result = []
    for text in texts:
        text = word2Sequence(text)[:MAX_LEN]
        text = torch.LongTensor(text)
        result.append(text)
    return pad_sequence(result,batch_first=True, padding_value=0)
    
padding(df['clean'])        

